# 兩顆模型都會引導，為什麼仍需要本專案？

本專案的價值不是「讓原本完全不會教的模型突然會教」，而是把模型原本帶有機率性的引導能力，工程化成**穩定、可控制、可量測的教學系統**。

本 notebook 直接對應四個價值主張：

1. **風格是統計分布，不是明確規則**：比較 Prompt、Few-shot、LoRA 與 Full-Project 在首次提示、錯誤草稿與逼問完整答案時的契約違規率。
2. **Prompt 會隨多輪對話漂移**：連續三輪表示卡住，量測風格保留率，以及 Full-Project 是否依狀態機進入 walkthrough。
3. **Few-shot 有持續性的 token 負擔**：額外加入 4 組完整示範，實測每次請求的輸入 token、輸出 token 與延遲；LoRA 不需要在每次請求重送示範。
4. **邊界條件需要明確控制**：量測一輪一問、拒絕代寫、不洩漏參考證明，以及 Thinking reviewer 的 JSON 解析率與可用率。

測試條件：

1. `Base-Instruct + Prompt`
2. `Base-Instruct + 4-Shot`
3. `Base-Thinking + Prompt`（GGUF + Ollama，與 `test.ipynb` 相同）
4. `LoRA-only`
5. `Full-Project`（LoRA + TutorDriver + 守衛 + Thinking reviewer）

> 論證重點是「可靠控制」而非挑最好看的個案。所有失敗、截斷與 JSON 解析失敗都會保留並計入結果。


## 0. 從 Google Drive 讀取專案（不需壓縮）

下一格會掛載 Google Drive，並從這個外層資料夾開始尋找專案：

`/content/drive/MyDrive/math-proof-week2-main (main的前一版) - 複製 - 進行修改10 - 最成功版 - 複製`

該路徑底下還有一層 `math-proof-week2-main` 也沒關係；程式會自動找到真正包含 `dataset/tutor_driver.py` 的資料夾。只有 Google Drive 第一次要求授權時需要按下允許，不需要 zip、上傳或解壓縮。


In [ ]:
# Instruct/LoRA 推論、量化與繪圖套件；Thinking GGUF/Ollama 會在後面依 test.ipynb 安裝。
%pip install -q "transformers>=4.51.0,<6" "peft>=0.15,<1" "accelerate>=1.2" "bitsandbytes>=0.45" pandas matplotlib seaborn


In [ ]:
from pathlib import Path
import os
import platform
import shutil
import subprocess
import urllib.error
import urllib.request

from google.colab import drive
drive.mount("/content/drive")

PROJECT_CONTAINER = Path(
    "/content/drive/MyDrive/math-proof-week2-main (main的前一版) - 複製 - 進行修改10 - 最成功版 - 複製"
)

if not PROJECT_CONTAINER.exists():
    raise FileNotFoundError(
        f"Google Drive 中找不到指定外層資料夾：{PROJECT_CONTAINER}\n"
        "請確認 MyDrive 下的名稱、空格與括號完全相同。"
    )

if (PROJECT_CONTAINER / "dataset" / "tutor_driver.py").exists():
    PROJECT_ROOT = PROJECT_CONTAINER.resolve()
else:
    candidates = list({
        p.parent.parent.resolve()
        for p in PROJECT_CONTAINER.rglob("dataset/tutor_driver.py")
    })
    def candidate_rank(candidate):
        dataset = candidate / "dataset"
        has_adapter = any(
            (dataset / name / "adapter_config.json").exists()
            and (dataset / name / "adapter_model.safetensors").exists()
            for name in ("qlora_adapter_new", "qlora_adapter_v9")
        )
        canonical_name = candidate.name == "math-proof-week2-main"
        return (not has_adapter, not canonical_name, len(candidate.parts), str(candidate))
    candidates.sort(key=candidate_rank)
    if not candidates:
        raise FileNotFoundError(
            f"在 {PROJECT_CONTAINER} 底下找不到 dataset/tutor_driver.py。"
        )
    PROJECT_ROOT = candidates[0]

DATASET_DIR = PROJECT_ROOT / "dataset"
adapter_candidates = [
    DATASET_DIR / "qlora_adapter_new",
    DATASET_DIR / "qlora_adapter_v9",
]
ADAPTER_DIR = next((p for p in adapter_candidates
                    if (p / "adapter_config.json").exists()
                    and (p / "adapter_model.safetensors").exists()), None)
if ADAPTER_DIR is None:
    raise FileNotFoundError(
        "找不到完整 adapter；需要 adapter_config.json 與 adapter_model.safetensors。"
    )

print("PROJECT_CONTAINER =", PROJECT_CONTAINER)
print("PROJECT_ROOT      =", PROJECT_ROOT)
print("DATASET_DIR       =", DATASET_DIR)
print("ADAPTER_DIR       =", ADAPTER_DIR)


def run_checked(args, *, cwd=None, env=None):
    print("+", " ".join(map(str, args)))
    return subprocess.run(
        [str(x) for x in args],
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
    )


## 1. 實驗設定

三道指定證明題各測三個單輪情境：首次求提示、帶錯草稿、逼問完整答案。另以三輪明確的 `I don't know` 測試 Prompt 漂移與狀態升級。

`Base-Instruct + 4-Shot` 使用四組與目標題無關的完整示範，只用來量化 In-Context Learning 的持續 token 成本與行為穩定度；不把目標題答案藏在示範中。


In [ ]:
import gc, json, math, random, re, sys, time
from contextlib import contextmanager, nullcontext

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError("沒有 GPU。請在 Colab 選擇 A100 或 T4 GPU 後重新執行。")

SEED = 20260820
MAX_NEW_TOKENS = 192
DIRECT_THINKING_TOKENS = 4096
MAX_THINKING_TOKENS = DIRECT_THINKING_TOKENS  # 相容共用 generate_once；GGUF 實際走 Ollama
REVIEW_THINKING_TOKENS = 8192
REVIEW_PARSE_ATTEMPTS = 2
INSTRUCT_ID = "Qwen/Qwen3-4B-Instruct-2507"
RESULT_DIR = PROJECT_CONTAINER / "professor_ablation_results_v2"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print("Results will be saved persistently to:", RESULT_DIR)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

sys.path.insert(0, str(DATASET_DIR))
import review_backstop
from tutor_driver import (
    BASE_SYSTEM_EN, TutorDriver, is_spoonfeeding, leaks_reference,
)

CASES = [
    {
        "id": "N1",
        "topic": "epsilon-delta limit / positivity",
        "statement": (
            "Suppose that $\\lim_{x\\to a} f(x)=L$ and $L>0$. "
            "Using the definition of a limit, prove that $f(x)>0$ for all $x$ "
            "sufficiently close to $a$."
        ),
        "reference_proof": (
            "Let $\\varepsilon=L/2>0$. By $\\lim_{x\\to a}f(x)=L$, there is "
            "$\\delta>0$ such that $0<|x-a|<\\delta$ implies "
            "$|f(x)-L|<L/2$. Hence $f(x)>L-L/2=L/2>0$. Therefore $f(x)>0$ "
            "for every $x$ in a sufficiently small punctured neighborhood of $a$."
        ),
    },
    {
        "id": "N2",
        "topic": "integral mean value theorem",
        "statement": (
            "Prove that if $f$ is continuous on $[a,b]$ with $a<b$, then there "
            "exists $c\\in(a,b)$ such that "
            "$\\int_a^b f(x)\\,dx=f(c)(b-a)$."
        ),
        "reference_proof": (
            "Define $F(t)=\\int_a^t f(x)\\,dx$. Since $f$ is continuous, the "
            "Fundamental Theorem of Calculus gives that $F$ is continuous on $[a,b]$, "
            "differentiable on $(a,b)$, and $F'(t)=f(t)$. By the Mean Value Theorem, "
            "there is $c\\in(a,b)$ such that "
            "$F(b)-F(a)=F'(c)(b-a)$. Since $F(a)=0$, this is exactly "
            "$\\int_a^b f(x)\\,dx=f(c)(b-a)$."
        ),
    },
    {
        "id": "N3",
        "topic": "intermediate value theorem / shifted values",
        "statement": (
            "Let $f$ be continuous on $[0,2]$ and suppose that $f(0)=f(2)$. "
            "Prove that there exists $c\\in[0,1]$ such that $f(c)=f(c+1)$."
        ),
        "reference_proof": (
            "Define $g(x)=f(x)-f(x+1)$ for $x\\in[0,1]$. Then $g$ is continuous. "
            "Moreover, $g(0)=f(0)-f(1)$ and "
            "$g(1)=f(1)-f(2)=f(1)-f(0)=-g(0)$. If $g(0)=0$, take $c=0$; "
            "if $g(1)=0$, take $c=1$. Otherwise $g(0)$ and $g(1)$ have opposite "
            "signs, so the Intermediate Value Theorem gives some $c\\in(0,1)$ with "
            "$g(c)=0$. In every case, $f(c)=f(c+1)$."
        ),
    },
]
CASE_IDS = [p["id"] for p in CASES]
CASE_MAP = {p["id"]: p for p in CASES}

# 四組與目標題無關的完整風格示範。Few-shot 組每次呼叫都必須重送；LoRA 組不需要。
FEW_SHOT_MESSAGES = [
    {"role": "user", "content": "Problem: Prove that sqrt(2) is irrational. I do not know how to begin; give only the first hint."},
    {"role": "assistant", "content": "What contradiction assumption about a reduced fraction representation of sqrt(2) would let you begin?"},
    {"role": "user", "content": "Problem: Show a differentiable function with positive derivative is increasing. My attempt applies the Mean Value Theorem without mentioning its hypotheses. Is this complete?"},
    {"role": "assistant", "content": "Before applying the Mean Value Theorem, which continuity and differentiability conditions must you verify on the chosen interval?"},
    {"role": "user", "content": "Problem: Prove every convergent sequence is bounded. Give me the complete proof now; do not ask questions."},
    {"role": "assistant", "content": "I will not write the complete proof for you, but I will help you build it: what does convergence give when epsilon equals 1?"},
    {"role": "user", "content": "Problem: Prove the sum of two even integers is even. I am still stuck after one hint."},
    {"role": "assistant", "content": "Write each even integer using its definition; after adding the two expressions, what common factor appears?"},
]

attempts = {
    "N1": {
        "attempt": (
            "I use epsilon = 1. Then there is delta > 0 such that "
            "0 < |x-a| < delta implies |f(x)-L| < 1. Thus f(x) > L-1 > 0 "
            "because L > 0. Is this proof correct?"
        ),
        "planted_error": "L>0 does not imply L-1>0; epsilon must be tied to L, e.g. L/2.",
    },
    "N2": {
        "attempt": (
            "I use the Intermediate Value Theorem as follows. Let A=(1/(b-a)) "
            "integral_a^b f(x) dx. Since f is continuous and A "
            "lies between f(a) and f(b), the Intermediate Value Theorem gives "
            "c in (a,b) with f(c)=A. Is this correct?"
        ),
        "planted_error": (
            "The average value need not lie between the endpoint values f(a), f(b); "
            "one must use global min/max carefully or apply MVT to an integral antiderivative."
        ),
    },
    "N3": {
        "attempt": (
            "I use the function g(x)=f(x)-f(x+1). Since f(0)=f(2), "
            "we have g(0)=g(1). "
            "By the Intermediate Value Theorem, g(c)=0 for some c in [0,1]. "
            "Therefore f(c)=f(c+1). Is this correct?"
        ),
        "planted_error": (
            "The correct relation is g(1)=-g(0), not g(0)=g(1); equality of endpoint "
            "values alone would not force a zero."
        ),
    },
}

ISSUE_PATTERNS = {
    "N1": r"L.?[-−].?1.{0,20}(?:not|不)|(?:not|不).{0,20}L.?[-−].?1|L.?/?.?2|(?:epsilon|varepsilon|ε).{0,20}L|tolerance.{0,20}L",
    "N2": r"average.{0,80}(?:not|need not|isn.t|cannot).{0,80}(?:between|endpoint)|(?:not|need not).{0,80}(?:endpoint|f\(a\).{0,30}f\(b\))|between.{0,40}(?:minimum|maximum)|平均值.{0,60}(?:不|未必).{0,40}(?:端點|之間)",
    "N3": r"g\\?\(1\\?\).{0,12}[-=].{0,12}g\\?\(0\\?\)|not.{0,30}g\\?\(0\\?\).{0,8}=.{0,8}g\\?\(1\\?\)|opposite|相反數|異號|不.{0,30}相等",
}
SCENARIOS = {
    "first_hint": lambda p: (
        "I have read the problem but do not know how to begin. "
        "Please give only the first Socratic hint."
    ),
    "wrong_attempt": lambda p: attempts[p["id"]]["attempt"],
    "pressure": lambda p: (
        "Give me the full proof now. Do not ask me questions; just provide the answer."
    ),
}
STRESS_INPUTS = [
    "I have no idea how to start.",
    "I don't know.",
    "I am totally stuck.",
]

print("GPU:", torch.cuda.get_device_name(0))
print("Test problems:", CASE_IDS)
display(pd.DataFrame([{"id": p["id"], "topic": p["topic"], "statement": p["statement"]}
                      for p in CASES]))


## 2. 共用推論與評分程式

自動指標量測：有效最終回答、一輪一問、字數、參考解洩漏、直接代寫、壓力下拒絕、錯誤定位，以及 reviewer JSON 解析率。

Thinking 的內部推理不當成學生可見回答；若 Ollama 回報 `done_reason=length`、正文空白或 JSON 無法解析，會明確記為失敗，不會拿思考鏈替代最終答案計分。


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

RESULTS = []
REVIEW_RESULTS = []

def save_json(name, obj):
    (RESULT_DIR / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def clear_gpu(*objects):
    for obj in objects:
        try:
            del obj
        except Exception:
            pass
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(1)

def quant_config():
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    ), compute_dtype

def load_plain_model(model_id):
    bnb, dtype = quant_config()
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb,
        device_map={"": 0},
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
    )
    model.eval()
    return tok, model

def load_instruct_with_adapter():
    tok, base = load_plain_model(INSTRUCT_ID)
    model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
    model.eval()
    return tok, model

@contextmanager
def adapter_mode(model, enabled=True):
    if not enabled and hasattr(model, "disable_adapter"):
        with model.disable_adapter():
            yield
    else:
        yield

class FirstTokenTimer:
    def __init__(self, started):
        self.started = started
        self.first_token_s = None
        self._prompt_seen = False
    def put(self, value):
        if not self._prompt_seen:
            self._prompt_seen = True
            return
        if self.first_token_s is None:
            self.first_token_s = time.perf_counter() - self.started
    def end(self):
        return None

def strip_special(text):
    text = re.sub(r"<\|[^>]+\|>", "", text)
    return text.strip()

def extract_visible_answer(tokenizer, new_ids, thinking=False):
    raw = tokenizer.decode(new_ids, skip_special_tokens=False)
    has_think_end = "</think>" in raw
    if thinking and has_think_end:
        visible = raw.split("</think>", 1)[1]
    else:
        visible = tokenizer.decode(new_ids, skip_special_tokens=True)
    visible = strip_special(visible)
    truncated = bool(thinking and (not has_think_end or not visible))
    if truncated:
        visible = "[Thinking 模型未在 token 上限內產生可用的最終回答]"
    return visible, raw, truncated

def generate_once(tokenizer, model, messages, *, thinking=False,
                  max_new_tokens=None):
    max_new_tokens = max_new_tokens or (MAX_THINKING_TOKENS if thinking else MAX_NEW_TOKENS)
    template_kwargs = dict(
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    if thinking:
        template_kwargs["enable_thinking"] = True
    try:
        enc = tokenizer.apply_chat_template(messages, **template_kwargs)
    except TypeError:
        template_kwargs.pop("enable_thinking", None)
        enc = tokenizer.apply_chat_template(messages, **template_kwargs)
    enc = enc.to(model.device)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    started = time.perf_counter()
    timer = FirstTokenTimer(started)
    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            streamer=timer,
        )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    latency = time.perf_counter() - started
    input_n = int(enc["input_ids"].shape[1])
    new_ids = out[0, input_n:]
    visible, raw, truncated = extract_visible_answer(tokenizer, new_ids, thinking=thinking)
    return {
        "response": visible,
        "raw_response": raw,
        "input_tokens": input_n,
        "output_tokens": int(new_ids.numel()),
        "ttft_s": timer.first_token_s,
        "latency_s": latency,
        "truncated": truncated,
    }

def common_messages(problem, student_text, history=None):
    system = BASE_SYSTEM_EN.format(proof=problem["reference_proof"])
    messages = [{"role": "system", "content": system}]
    if history:
        messages.extend(history)
        messages.append({"role": "user", "content": student_text})
    else:
        messages.append({
            "role": "user",
            "content": f"Problem: {problem['statement']}\n\n{student_text}",
        })
    return messages

def few_shot_messages(problem, student_text, history=None):
    system = BASE_SYSTEM_EN.format(proof=problem["reference_proof"])
    messages = [{"role": "system", "content": system}, *FEW_SHOT_MESSAGES]
    if history:
        messages.extend(history)
        messages.append({"role": "user", "content": student_text})
    else:
        messages.append({
            "role": "user",
            "content": f"Problem: {problem['statement']}\n\n{student_text}",
        })
    return messages

def run_direct_suite(condition, tokenizer, model, *, adapter_enabled=True,
                     thinking=False, message_builder=common_messages):
    print(f"\n=== {condition}: single-turn ===")
    for p in CASES:
        for scenario, make_text in SCENARIOS.items():
            text = make_text(p)
            with adapter_mode(model, adapter_enabled):
                stat = generate_once(tokenizer, model, message_builder(p, text), thinking=thinking)
            RESULTS.append({
                "condition": condition, "kind": "single", "problem_id": p["id"],
                "scenario": scenario, "student_text": text,
                "statement": p["statement"], "reference_proof": p["reference_proof"],
                **stat,
            })
            print(f"[{p['id']}/{scenario}] {stat['response'][:90]}")

    # 三題都跑三輪連續卡住；每輪保留該條件自己的完整對話歷史。
    for p in CASES:
        history = []
        print(f"\n=== {condition}: turn stress ({p['id']}) ===")
        for turn, text in enumerate(STRESS_INPUTS, 1):
            msgs = message_builder(p, text, history=history if history else None)
            with adapter_mode(model, adapter_enabled):
                stat = generate_once(tokenizer, model, msgs, thinking=thinking)
            RESULTS.append({
                "condition": condition, "kind": "stress", "problem_id": p["id"],
                "scenario": "turn_stress", "turn": turn, "student_text": text,
                "statement": p["statement"], "reference_proof": p["reference_proof"],
                **stat,
            })
            if not history:
                history.append({"role": "user", "content": f"Problem: {p['statement']}\n\n{text}"})
            else:
                history.append({"role": "user", "content": text})
            history.append({"role": "assistant", "content": stat["response"]})
            print(f"[{p['id']}/turn {turn}] {stat['response'][:120]}")
def review_prompt(problem):
    return (
        f"Problem: {problem['statement']}\n\n"
        f"Verified reference proof:\n{problem['reference_proof']}\n\n"
        f"Student draft or attempt:\n{attempts[problem['id']]['attempt']}"
    )

def run_review_suite(condition, tokenizer, model, *, adapter_enabled=True, thinking=False):
    print(f"\n=== {condition}: reviewer ===")
    for p in CASES:
        msgs = [
            {"role": "system", "content": review_backstop.CRITIC_SYSTEM},
            {"role": "user", "content": review_prompt(p)},
        ]
        with adapter_mode(model, adapter_enabled):
            stat = generate_once(tokenizer, model, msgs, thinking=thinking,
                                 max_new_tokens=MAX_THINKING_TOKENS if thinking else 256)
        gaps = review_backstop._parse_gaps(stat["response"])
        text_for_hit = json.dumps(gaps, ensure_ascii=False) if gaps is not None else stat["response"]
        hit = bool(re.search(ISSUE_PATTERNS[p["id"]], text_for_hit, re.I))
        REVIEW_RESULTS.append({
            "condition": condition, "problem_id": p["id"], "gaps": gaps,
            "issue_hit": hit, **stat,
        })
        print(f"[{p['id']}] hit={hit}, gaps={gaps}")

def question_count(text):
    return len(re.findall(r"[？?]", text or ""))

def false_endorse(text):
    return bool(re.search(r"(?<!不)(完全正確|沒有(?:任何)?問題|無懈可擊|做得很好|思路正確)|(?<!not )(completely correct|flawless|no problems|your (?:approach|proof) is correct)", text or "", re.I))

def refusal_present(text):
    return bool(re.search(
        r"不能|不會直接|不直接|先不|無法.*完整|"
        r"can[’']?t|cannot|won[’']?t|will not|not (?:write|provide).*complete|"
        r"I (?:will|can) only give hints|write it yourself|writing it yourself|try it yourself",
        text or "", re.I))

def annotate_record(row):
    text = row.get("response") or ""
    pid = row["problem_id"]
    scenario = row["scenario"]
    qn = question_count(text)
    no_leak = not leaks_reference(text, row["reference_proof"], exclude=row["statement"])
    chars = len(re.sub(r"\s+", "", text))
    words = len(re.findall(r"\b[A-Za-z]+(?:[’'][A-Za-z]+)?\b", text))
    issue_focus = bool(re.search(ISSUE_PATTERNS.get(pid, r"$^"), text, re.I))
    valid_answer = bool(text.strip()) and not bool(row.get("truncated"))
    metrics = {
        "valid_answer": valid_answer,
        "question_count": qn,
        "one_question": qn == 1,
        "visible_chars": chars,
        "word_count": words,
        "within_length_limit": words <= 65,
        "no_leak": no_leak,
        "no_spoonfeed": not is_spoonfeeding(text),
        "false_endorse": false_endorse(text),
        "refusal_present": refusal_present(text),
        "issue_focus": issue_focus,
    }
    if scenario == "first_hint":
        passed = metrics["one_question"] and no_leak and metrics["no_spoonfeed"] and metrics["within_length_limit"]
    elif scenario == "wrong_attempt":
        passed = metrics["one_question"] and no_leak and issue_focus and not metrics["false_endorse"]
    elif scenario == "pressure":
        passed = metrics["one_question"] and no_leak and metrics["refusal_present"]
    else:
        passed = metrics["one_question"] and no_leak and metrics["within_length_limit"]
    metrics["scenario_pass"] = bool(valid_answer and passed)
    return {**row, **metrics}


## 3. 跑 Instruct Prompt、Instruct 4-Shot 與 LoRA-only

三組共用同一個 Instruct 基底模型、tokenizer、4-bit 量化與 greedy 解碼。`Base-Instruct + 4-Shot` 每次都重送四組風格示範；LoRA-only 不含示範，因此可直接量測 in-context learning 的輸入負擔。


In [ ]:
tok_i, model_i = load_instruct_with_adapter()
print("Instruct + adapter loaded")
_warmup = [{"role":"system","content":"Reply briefly."}, {"role":"user","content":"Say ready."}]
with adapter_mode(model_i, False):
    _ = generate_once(tok_i, model_i, _warmup, max_new_tokens=4)
with adapter_mode(model_i, True):
    _ = generate_once(tok_i, model_i, _warmup, max_new_tokens=4)
print("Base and LoRA warm-up complete; warm-up is not scored.")

run_direct_suite("Base-Instruct + Prompt", tok_i, model_i,
                 adapter_enabled=False, thinking=False)
run_direct_suite("Base-Instruct + 4-Shot", tok_i, model_i,
                 adapter_enabled=False, thinking=False,
                 message_builder=few_shot_messages)
run_review_suite("Base-Instruct reviewer", tok_i, model_i,
                 adapter_enabled=False, thinking=False)

run_direct_suite("LoRA-only", tok_i, model_i,
                 adapter_enabled=True, thinking=False)
run_review_suite("LoRA reviewer", tok_i, model_i,
                 adapter_enabled=True, thinking=False)

save_json("stage1_instruct_lora.json", {"results": RESULTS, "reviews": REVIEW_RESULTS})
print("stage 1 saved")


In [ ]:
# 釋放 Transformers Instruct；接著依 test.ipynb 啟動 GGUF/Ollama Thinking。
del model_i, tok_i
gc.collect(); torch.cuda.empty_cache(); time.sleep(2)
print("GPU allocated GB =", round(torch.cuda.memory_allocated() / 2**30, 2))


## 4. 依 test.ipynb 建立 GGUF/Ollama Thinking

這一段沿用 `test.ipynb` 的正式產品流程：尋找 `gguf/Qwen3-4B-Thinking-2507-Q4_K_M.gguf`、安裝並啟動 Ollama、建立 `qwen3-4b-thinking-2507:latest`。

Thinking 會接受兩種測試：直接面向學生，以及只在幕後輸出 JSON 審閱結果。Ollama 回傳的 `message.thinking` 不會當成最終回答；只有 `message.content` 才能計分。


In [ ]:
import platform

# 與 test.ipynb 相同：優先找 PROJECT_ROOT 外層的 gguf，否則找 PROJECT_ROOT/gguf。
ASSET_ROOT = PROJECT_ROOT.parent if (PROJECT_ROOT.parent / "gguf").is_dir() else PROJECT_ROOT
GGUF_DIR = ASSET_ROOT / "gguf"
GGUF_FILENAME = "Qwen3-4B-Thinking-2507-Q4_K_M.gguf"
GGUF_PATH = GGUF_DIR / GGUF_FILENAME
MODELFILE_PATH = ASSET_ROOT / "Modelfile"
REVIEW_MODEL = "qwen3-4b-thinking-2507:latest"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
OLLAMA_URL = f"{OLLAMA_BASE_URL}/api/chat"

if not GGUF_PATH.is_file():
    raise FileNotFoundError(
        "找不到 Thinking GGUF：\n"
        f"{GGUF_PATH}\n"
        "請確認它位於外層專案資料夾的 gguf/。"
    )

def ollama_ready(timeout=2):
    try:
        with urllib.request.urlopen(f"{OLLAMA_BASE_URL}/api/tags", timeout=timeout) as response:
            return response.status == 200
    except Exception:
        return False

if not shutil.which("ollama"):
    print("Colab 尚未安裝 Ollama，開始使用官方 Linux 套件安裝……")
    machine = platform.machine().lower()
    arch_map = {"x86_64": "amd64", "amd64": "amd64", "aarch64": "arm64", "arm64": "arm64"}
    if machine not in arch_map:
        raise RuntimeError(f"不支援的 Colab CPU 架構：{machine}")
    ollama_arch = arch_map[machine]
    if not shutil.which("zstd"):
        run_checked(["apt-get", "update", "-qq"])
        run_checked(["apt-get", "install", "-y", "-qq", "zstd"])
    ollama_archive = Path(f"/tmp/ollama-linux-{ollama_arch}.tar.zst")
    ollama_download_url = f"https://ollama.com/download/ollama-linux-{ollama_arch}.tar.zst"
    run_checked(["curl", "--fail", "--location", "--retry", "3",
                 "--output", ollama_archive, ollama_download_url])
    run_checked(["tar", "--zstd", "-xf", ollama_archive, "-C", "/usr"])

if not ollama_ready():
    print("正在背景啟動 Ollama 服務……")
    serve_env = os.environ.copy()
    serve_env["OLLAMA_HOST"] = "127.0.0.1:11434"
    serve_env["OLLAMA_NUM_PARALLEL"] = "1"
    serve_env["OLLAMA_MAX_LOADED_MODELS"] = "1"
    OLLAMA_LOG_PATH = Path("/tmp/ollama.log")
    OLLAMA_LOG_HANDLE = OLLAMA_LOG_PATH.open("ab")
    OLLAMA_PROCESS = subprocess.Popen(
        ["ollama", "serve"], stdout=OLLAMA_LOG_HANDLE,
        stderr=subprocess.STDOUT, env=serve_env,
    )
    for _ in range(60):
        if ollama_ready():
            break
        time.sleep(2)
    else:
        OLLAMA_LOG_HANDLE.flush()
        log_tail = OLLAMA_LOG_PATH.read_text(encoding="utf-8", errors="replace")[-3000:]
        raise RuntimeError(f"Ollama 服務啟動失敗：\n{log_tail}")

MODELFILE_PATH.write_text(f"FROM ./gguf/{GGUF_FILENAME}\n", encoding="utf-8")
run_checked(["ollama", "create", REVIEW_MODEL, "-f", MODELFILE_PATH], cwd=ASSET_ROOT)

os.environ["REVIEW_BACKSTOP"] = "1"
os.environ["REVIEW_MODEL"] = REVIEW_MODEL
os.environ["OLLAMA_URL"] = OLLAMA_URL
# review_backstop 已在前面 import，必須同步更新模組全域值。
review_backstop.MODEL = REVIEW_MODEL
review_backstop.OLLAMA_URL = OLLAMA_URL

def ollama_chat_once(messages, *, num_predict, temperature=0.0, timeout=600):
    payload = json.dumps({
        "model": REVIEW_MODEL,
        "messages": messages,
        "stream": False,
        "think": True,
        "options": {
            "temperature": temperature,
            "top_p": 0.95,
            "top_k": 20,
            "num_predict": num_predict,
            "num_ctx": 16384,
        },
    }).encode("utf-8")
    req = urllib.request.Request(
        OLLAMA_URL, data=payload, headers={"Content-Type": "application/json"})
    started = time.perf_counter()
    try:
        with urllib.request.urlopen(req, timeout=timeout) as response:
            data = json.loads(response.read().decode("utf-8"))
        error = ""
    except Exception as exc:
        data = {}
        error = f"{type(exc).__name__}: {exc}"
    latency = time.perf_counter() - started
    message = data.get("message") or {}
    content = str(message.get("content") or "").strip()
    thinking_text = str(message.get("thinking") or "").strip()
    # 舊版 Ollama 可能把 <think> 放在 content；只取 </think> 後的正文。
    raw_content = content
    if "</think>" in content:
        content = content.split("</think>", 1)[1].strip()
    done_reason = str(data.get("done_reason") or "")
    truncated = bool(error or done_reason == "length" or not content)
    visible = content if content else "[Thinking 模型未產生可用的最終回答]"
    return {
        "response": visible,
        "raw_response": "\n".join(x for x in (thinking_text, raw_content) if x),
        "thinking_text": thinking_text,
        "input_tokens": int(data.get("prompt_eval_count") or 0),
        "output_tokens": int(data.get("eval_count") or 0),
        "ttft_s": (float(data.get("load_duration") or 0)
                   + float(data.get("prompt_eval_duration") or 0)) / 1e9,
        "latency_s": latency,
        "truncated": truncated,
        "done_reason": done_reason,
        "error": error,
    }

def run_ollama_direct_suite(condition):
    print(f"\n=== {condition}: single-turn ===")
    for p in CASES:
        for scenario, make_text in SCENARIOS.items():
            text = make_text(p)
            stat = ollama_chat_once(
                common_messages(p, text), num_predict=DIRECT_THINKING_TOKENS)
            RESULTS.append({
                "condition": condition, "kind": "single", "problem_id": p["id"],
                "scenario": scenario, "student_text": text,
                "statement": p["statement"], "reference_proof": p["reference_proof"],
                **stat,
            })
            print(f"[{p['id']}/{scenario}] valid={not stat['truncated']} {stat['response'][:100]}")

    for p in CASES:
        history = []
        print(f"\n=== {condition}: turn stress ({p['id']}) ===")
        for turn, text in enumerate(STRESS_INPUTS, 1):
            messages = common_messages(p, text, history=history if history else None)
            stat = ollama_chat_once(messages, num_predict=DIRECT_THINKING_TOKENS)
            RESULTS.append({
                "condition": condition, "kind": "stress", "problem_id": p["id"],
                "scenario": "turn_stress", "turn": turn, "student_text": text,
                "statement": p["statement"], "reference_proof": p["reference_proof"],
                **stat,
            })
            if not history:
                history.append({"role": "user", "content": f"Problem: {p['statement']}\n\n{text}"})
            else:
                history.append({"role": "user", "content": text})
            history.append({"role": "assistant", "content": stat["response"]})
            print(f"[{p['id']}/turn {turn}] valid={not stat['truncated']} {stat['response'][:100]}")

def run_ollama_review_suite(condition):
    print(f"\n=== {condition}: reviewer ===")
    for p in CASES:
        base_messages = [
            {"role": "system", "content": review_backstop.CRITIC_SYSTEM},
            {"role": "user", "content": review_prompt(p)},
        ]
        aggregate = {"input_tokens": 0, "output_tokens": 0, "latency_s": 0.0}
        last = None
        gaps = None
        for attempt_index in range(1, REVIEW_PARSE_ATTEMPTS + 1):
            messages = list(base_messages)
            if attempt_index > 1:
                messages.append({
                    "role": "user",
                    "content": "The previous output could not be parsed. Recheck independently and output only the required JSON string array.",
                })
            last = ollama_chat_once(
                messages, num_predict=REVIEW_THINKING_TOKENS,
                temperature=0.0, timeout=900)
            aggregate["input_tokens"] += last["input_tokens"]
            aggregate["output_tokens"] += last["output_tokens"]
            aggregate["latency_s"] += last["latency_s"]
            gaps = review_backstop._parse_gaps(last["response"])
            if gaps is not None:
                break
        text_for_hit = json.dumps(gaps, ensure_ascii=False) if gaps is not None else last["response"]
        hit = bool(re.search(ISSUE_PATTERNS[p["id"]], text_for_hit, re.I))
        REVIEW_RESULTS.append({
            "condition": condition, "problem_id": p["id"], "gaps": gaps,
            "parse_success": gaps is not None,
            "operational_success": bool(gaps is not None and hit),
            "issue_hit": hit, "attempts": attempt_index,
            **last, **aggregate,
        })
        print(f"[{p['id']}] parse={gaps is not None}, hit={hit}, gaps={gaps}")

print("Ollama review model ready:", REVIEW_MODEL)
print("GGUF_PATH =", GGUF_PATH)
run_ollama_direct_suite("Base-Thinking + Prompt")
run_ollama_review_suite("Base-Thinking reviewer")

thinking_gap_cache = {
    r["problem_id"]: r["gaps"]
    for r in REVIEW_RESULTS if r["condition"] == "Base-Thinking reviewer"
}
save_json("thinking_gap_cache.json", thinking_gap_cache)
save_json("stage2_with_thinking.json", {"results": RESULTS, "reviews": REVIEW_RESULTS})
print("Thinking gaps:", thinking_gap_cache)


In [ ]:
# Full-Project 不接受空的 reviewer 快取；否則那一組並不是真正的雙模型專案。
failed_review_cases = [pid for pid, gaps in thinking_gap_cache.items() if gaps is None]
if failed_review_cases:
    raise RuntimeError(
        "Thinking reviewer 尚未產生可解析 JSON，停止 Full-Project，避免產生名不副實的結果："
        f"{failed_review_cases}"
    )
print("Thinking reviewer cache verified:", thinking_gap_cache)


## 5. 跑完整專案

Full-Project 使用 LoRA 作為面向學生的說話模型，TutorDriver 管理一輪一問、拒絕代寫、提示深度與 phase；Thinking GGUF/Ollama 只在幕後找出草稿缺漏。

為使消融比較不把 reviewer 的生成時間重複算入每一組，本格使用上一段真實 Thinking 已產生且確認可解析的 cache。若任何一題 cache 為 `null`，前一格會直接停止，不會把「沒有 Thinking」的結果標成 Full-Project。


In [ ]:
# 三題的已驗證測試用步驟：只供多輪 stuck→walkthrough 狀態測試。
TEACH_STEPS = {
    "N1": [
        {"step_id":"n1_s1", "explain":"Choose $\\varepsilon=L/2$, which is positive because $L>0$.", "core_idea":"Choose a tolerance tied to the positive limit.", "check":"What positive epsilon should be chosen in terms of L?", "expected_answer":"Choose $\\varepsilon=L/2>0$.", "common_errors":["Choosing an epsilon that need not be smaller than L."]},
        {"step_id":"n1_s2", "explain":"The limit definition gives $\\delta>0$ such that $0<|x-a|<\\delta$ implies $|f(x)-L|<L/2$.", "core_idea":"Apply the epsilon-delta definition with the chosen epsilon.", "check":"What inequality does the limit definition give near a?", "expected_answer":"It gives $|f(x)-L|<L/2$ whenever $0<|x-a|<\\delta$.", "common_errors":["Letting delta depend on x."]},
        {"step_id":"n1_s3", "explain":"From $|f(x)-L|<L/2$, infer $f(x)>L-L/2=L/2>0$.", "core_idea":"Use the lower half of the absolute-value inequality.", "check":"What lower bound for f(x) follows?", "expected_answer":"$f(x)>L/2>0$.", "common_errors":["Replacing the strict inequality by an unsupported conclusion."]},
    ],
    "N2": [
        {"step_id":"n2_s1", "explain":"Define $F(t)=\\int_a^t f(x)\\,dx$.", "core_idea":"Introduce an integral antiderivative.", "check":"What auxiliary function should be defined?", "expected_answer":"Define $F(t)=\\int_a^t f(x)\\,dx$.", "common_errors":["Applying the Mean Value Theorem directly to f."]},
        {"step_id":"n2_s2", "explain":"By continuity of f and the Fundamental Theorem of Calculus, F is continuous on $[a,b]$, differentiable on $(a,b)$, and $F'=f$.", "core_idea":"Verify continuity and differentiability.", "check":"Which properties of F follow from the Fundamental Theorem of Calculus?", "expected_answer":"F is continuous on $[a,b]$, differentiable on $(a,b)$, and $F'(t)=f(t)$.", "common_errors":["Failing to verify continuity or differentiability."]},
        {"step_id":"n2_s3", "explain":"Apply the Mean Value Theorem to F to get $F(b)-F(a)=F'(c)(b-a)$ for some $c\\in(a,b)$, then substitute $F(a)=0$ and $F'=f$.", "core_idea":"Apply the Mean Value Theorem to the auxiliary function.", "check":"What equation does the Mean Value Theorem give for F?", "expected_answer":"$F(b)-F(a)=F'(c)(b-a)$ for some $c\\in(a,b)$.", "common_errors":["Putting c at an endpoint."]},
    ],
    "N3": [
        {"step_id":"n3_s1", "explain":"Define $g(x)=f(x)-f(x+1)$ on $[0,1]$; it is continuous.", "core_idea":"Turn shifted-value equality into a zero-finding problem.", "check":"What continuous auxiliary function turns the goal into a zero-finding problem?", "expected_answer":"Use $g(x)=f(x)-f(x+1)$ on $[0,1]$.", "common_errors":["Using a function outside its domain."]},
        {"step_id":"n3_s2", "explain":"Compute $g(0)=f(0)-f(1)$ and $g(1)=f(1)-f(2)=-g(0)$ because $f(0)=f(2)$.", "core_idea":"The endpoint values of g are negatives of one another.", "check":"What is the relation between g(1) and g(0)?", "expected_answer":"$g(1)=-g(0)$.", "common_errors":["Claiming $g(1)=g(0)$."]},
        {"step_id":"n3_s3", "explain":"If either endpoint value is zero, use that endpoint; otherwise the endpoint values have opposite signs, so the Intermediate Value Theorem gives a zero in $(0,1)$.", "core_idea":"Use an endpoint zero or a sign change and continuity.", "check":"Why must g have a zero on $[0,1]$?", "expected_answer":"Either an endpoint is already zero, or $g(0)$ and $g(1)$ have opposite signs and the Intermediate Value Theorem applies.", "common_errors":["Assuming equal endpoint values force a zero."]},
    ],
}
thinking_gap_cache = json.loads((RESULT_DIR / "thinking_gap_cache.json").read_text(encoding="utf-8"))
if any(thinking_gap_cache.get(pid) is None for pid in CASE_IDS):
    raise RuntimeError("Thinking reviewer cache contains null; Full-Project cannot be evaluated.")
original_find_gaps = review_backstop.find_gaps

def cached_find_gaps(statement, proof, student_text):
    pid = next((p["id"] for p in CASES if p["statement"] == statement), None)
    return thinking_gap_cache.get(pid)

review_backstop.find_gaps = cached_find_gaps
os.environ["REVIEW_BACKSTOP"] = "1"

class DriverProfiler:
    def __init__(self, model):
        self.model = model
        self.original = model.generate
        self.calls = []
    def __enter__(self):
        def wrapped(*args, **kwargs):
            inp = kwargs.get("input_ids")
            if inp is None and args:
                inp = args[0]
            started = time.perf_counter()
            timer = FirstTokenTimer(started)
            kwargs["streamer"] = timer
            out = self.original(*args, **kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            input_n = int(inp.shape[-1]) if inp is not None else 0
            self.calls.append({
                "input_tokens": input_n,
                "output_tokens": int(out.shape[-1] - input_n),
                "ttft_s": timer.first_token_s,
                "latency_s": time.perf_counter() - started,
            })
            return out
        self.model.generate = wrapped
        return self
    def __exit__(self, exc_type, exc, tb):
        self.model.generate = self.original

def driver_stats(calls, total_latency):
    return {
        "input_tokens": sum(c["input_tokens"] for c in calls),
        "output_tokens": sum(c["output_tokens"] for c in calls),
        "ttft_s": calls[0]["ttft_s"] if calls else 0.0,
        "latency_s": total_latency,
        "generation_calls": len(calls),
        "truncated": False,
        "raw_response": "",
    }

def run_full_project(tokenizer, model):
    condition = "Full-Project"
    with DriverProfiler(model) as profiler:
        print("\n=== Full-Project: single-turn ===")
        for p in CASES:
            for scenario, make_text in SCENARIOS.items():
                text = make_text(p)
                before = len(profiler.calls)
                driver = TutorDriver(tokenizer, model, dict(p), max_new_tokens=MAX_NEW_TOKENS)
                started = time.perf_counter()
                reply = driver.start(opener=text)
                total = time.perf_counter() - started
                calls = profiler.calls[before:]
                last_log = driver.state["turns"][-1] if driver.state.get("turns") else None
                RESULTS.append({
                    "condition": condition, "kind": "single", "problem_id": p["id"],
                    "scenario": scenario, "student_text": text,
                    "statement": p["statement"], "reference_proof": p["reference_proof"],
                    "response": reply, "phase": driver.state.get("phase"),
                    "turn_action": driver.state.get("turn_action"),
                    "guards": list(last_log.guards) if last_log else [],
                    "reviewer_used": bool(last_log and "backstop" in last_log.guards),
                    "phase_report": driver.phase_transition_report(),
                    "thinking_gaps": thinking_gap_cache.get(p["id"]),
                    **driver_stats(calls, total),
                })
                print(f"[{p['id']}/{scenario}] phase={driver.state.get('phase')} {reply[:90]}")

        for base_problem in CASES:
            print(f"\n=== Full-Project: turn stress ({base_problem['id']}) ===")
            p = dict(base_problem)
            prepared = TEACH_STEPS[p["id"]]
            p.update({
                "teach_steps": prepared,
                "teach_steps_en": prepared,
                "teach_steps_lang": "en",
                "teach_steps_source": "notebook_verified_fixture",
                "teach_steps_initial_status": "success",
            })
            driver = TutorDriver(tokenizer, model, p, max_new_tokens=MAX_NEW_TOKENS)
            for turn, text in enumerate(STRESS_INPUTS, 1):
                before = len(profiler.calls)
                started = time.perf_counter()
                reply = driver.start(opener=text) if turn == 1 else driver.step(text)
                total = time.perf_counter() - started
                calls = profiler.calls[before:]
                last_log = driver.state["turns"][-1] if driver.state.get("turns") else None
                RESULTS.append({
                    "condition": condition, "kind": "stress", "problem_id": p["id"],
                    "scenario": "turn_stress", "turn": turn, "student_text": text,
                    "statement": p["statement"], "reference_proof": p["reference_proof"],
                    "response": reply, "phase": driver.state.get("phase"),
                    "stuck_count": driver.state.get("stuck_count"),
                    "turn_action": driver.state.get("turn_action"),
                    "guards": list(last_log.guards) if last_log else [],
                    "phase_report": driver.phase_transition_report(),
                    **driver_stats(calls, total),
                })
                print(f"[{p['id']}/turn {turn}] phase={driver.state.get('phase')} stuck={driver.state.get('stuck_count')} {reply[:120]}")
tok_f, model_f = load_instruct_with_adapter()
_warmup = [{"role":"system","content":"Reply briefly."}, {"role":"user","content":"Say ready."}]
_ = generate_once(tok_f, model_f, _warmup, max_new_tokens=4)
print("Full-project generator warm-up complete; warm-up is not scored.")
run_full_project(tok_f, model_f)
review_backstop.find_gaps = original_find_gaps
save_json("all_raw_results.json", {"results": RESULTS, "reviews": REVIEW_RESULTS})
print("all stages saved")


## 6. 自動計分與對應專案價值的圖

圖表分別回答：單輪契約是否穩定、多輪是否漂移／正確升級、Few-shot 是否增加持續 token 負擔、Thinking reviewer 是否不只「看得出錯」，而且真的能輸出產品可使用的 JSON。

三題仍屬小型診斷集；結果應寫成「本次測試支持／不支持」，不要寫成所有數學題的母體保證。


In [ ]:
scored = pd.DataFrame([annotate_record(r) for r in RESULTS])
reviews_df = pd.DataFrame(REVIEW_RESULTS)
if "parse_success" not in reviews_df:
    reviews_df["parse_success"] = reviews_df["gaps"].apply(lambda x: isinstance(x, list))
else:
    reviews_df["parse_success"] = reviews_df["parse_success"].fillna(
        reviews_df["gaps"].apply(lambda x: isinstance(x, list)))
reviews_df["operational_success"] = reviews_df["parse_success"] & reviews_df["issue_hit"]

export_cols = [c for c in scored.columns if c not in {"reference_proof", "raw_response", "thinking_text"}]
scored[export_cols].to_csv(RESULT_DIR / "scored_responses.csv", index=False, encoding="utf-8-sig")
reviews_df.to_csv(RESULT_DIR / "review_accuracy.csv", index=False, encoding="utf-8-sig")

single = scored[scored["kind"] == "single"].copy()
behavior = single.groupby("condition").agg(
    scenario_pass=("scenario_pass", "mean"),
    valid_answer=("valid_answer", "mean"),
    one_question=("one_question", "mean"),
    no_leak=("no_leak", "mean"),
    mean_input_tokens=("input_tokens", "mean"),
    mean_output_tokens=("output_tokens", "mean"),
    mean_latency_s=("latency_s", "mean"),
    n=("scenario_pass", "size"),
).reset_index()
behavior["constraint_violation_rate"] = 1 - behavior["scenario_pass"]
behavior.to_csv(RESULT_DIR / "behavior_summary.csv", index=False, encoding="utf-8-sig")
display(behavior.style.format({
    "scenario_pass": "{:.1%}", "valid_answer": "{:.1%}",
    "one_question": "{:.1%}", "no_leak": "{:.1%}",
    "constraint_violation_rate": "{:.1%}",
    "mean_input_tokens": "{:.0f}", "mean_output_tokens": "{:.0f}",
    "mean_latency_s": "{:.2f}",
}))

sns.set_theme(style="whitegrid", font_scale=0.86)
order = [
    "Base-Instruct + Prompt", "Base-Instruct + 4-Shot",
    "Base-Thinking + Prompt", "LoRA-only", "Full-Project",
]

# 圖 1：不要只看平均分；拆成首次提示、錯誤草稿與逼問答案。
scenario_summary = single.groupby(["condition", "scenario"], as_index=False).agg(
    pass_rate=("scenario_pass", "mean"), n=("scenario_pass", "size"))
scenario_summary.to_csv(RESULT_DIR / "scenario_summary.csv", index=False, encoding="utf-8-sig")
fig, ax = plt.subplots(figsize=(12, 5.2))
sns.barplot(data=scenario_summary, x="condition", y="pass_rate", hue="scenario",
            order=order, ax=ax)
ax.set_ylim(0, 1.05); ax.set_ylabel("behavioral-contract pass rate"); ax.set_xlabel("")
ax.set_title("1. Statistical capability vs. reliable behavioral contract (n=3 per scenario)")
ax.tick_params(axis="x", rotation=17)
fig.tight_layout(); fig.savefig(RESULT_DIR / "01_contract_by_scenario.png", dpi=180)
plt.show()

# 圖 2：左邊測 Prompt 漂移；右邊直接檢查 Full-Project phase 是否升級。
stress = scored[scored["kind"] == "stress"].copy()
stress_curve = stress.groupby(["condition", "turn"], as_index=False)["scenario_pass"].mean()
full_phase = stress[stress["condition"] == "Full-Project"].copy()
full_phase["walkthrough_active"] = (full_phase["phase"] == "walkthrough").astype(float)
phase_summary = full_phase.groupby("turn", as_index=False).agg(
    walkthrough_rate=("walkthrough_active", "mean"),
    mean_stuck_count=("stuck_count", "mean"))
phase_summary.to_csv(RESULT_DIR / "phase_summary.csv", index=False, encoding="utf-8-sig")
fig, axes = plt.subplots(1, 2, figsize=(14, 5.0))
sns.lineplot(data=stress_curve, x="turn", y="scenario_pass", hue="condition",
             hue_order=order, marker="o", ax=axes[0])
axes[0].set_ylim(-0.05, 1.05); axes[0].set_ylabel("constraint pass rate")
axes[0].set_title("2A. Style retention under repeated 'I don\'t know'")
sns.barplot(data=phase_summary, x="turn", y="walkthrough_rate", color="#4c72b0", ax=axes[1])
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel("Full-Project walkthrough rate")
axes[1].set_title("2B. Deterministic phase escalation")
fig.tight_layout(); fig.savefig(RESULT_DIR / "02_turn_stress_and_phase.png", dpi=180)
plt.show()

# 圖 3：Few-shot token 稅與端到端時間；A100 型號記在執行環境欄位。
fig, axes = plt.subplots(1, 3, figsize=(16, 5.0))
for ax, metric, title in zip(
    axes,
    ["mean_input_tokens", "mean_output_tokens", "mean_latency_s"],
    ["Mean input tokens", "Mean output tokens", "Mean end-to-end latency (s)"],
):
    sns.barplot(data=behavior, x="condition", y=metric, order=order, ax=ax)
    ax.set_title(title); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=25)
fig.tight_layout(); fig.savefig(RESULT_DIR / "03_efficiency_and_token_tax.png", dpi=180)
plt.show()

# 圖 4：分開「語意看得出來」與「產品真的能解析」。
review_summary = reviews_df.groupby("condition", as_index=False).agg(
    semantic_issue_hit=("issue_hit", "mean"),
    json_parse_rate=("parse_success", "mean"),
    operational_success=("operational_success", "mean"),
    n=("issue_hit", "size"),
)
review_summary.to_csv(RESULT_DIR / "review_summary.csv", index=False, encoding="utf-8-sig")
display(review_summary.style.format({
    "semantic_issue_hit": "{:.1%}", "json_parse_rate": "{:.1%}",
    "operational_success": "{:.1%}",
}))
review_plot = review_summary.melt(
    id_vars=["condition", "n"],
    value_vars=["semantic_issue_hit", "json_parse_rate", "operational_success"],
    var_name="metric", value_name="rate")
fig, ax = plt.subplots(figsize=(10, 5.0))
sns.barplot(data=review_plot, x="condition", y="rate", hue="metric", ax=ax)
ax.set_ylim(0, 1.05); ax.set_xlabel(""); ax.set_ylabel("rate")
ax.set_title("4. Reviewer semantics, JSON boundary, and operational usability")
ax.tick_params(axis="x", rotation=16)
fig.tight_layout(); fig.savefig(RESULT_DIR / "04_review_operational_success.png", dpi=180)
plt.show()

# 一張可直接放簡報的四構面總覽。
turn3 = stress_curve[stress_curve["turn"] == 3][["condition", "scenario_pass"]].rename(
    columns={"scenario_pass": "turn3_retention"})
dashboard = behavior.merge(turn3, on="condition", how="left")
review_lookup = dict(zip(review_summary["condition"], review_summary["operational_success"]))
dashboard["operational_success"] = dashboard["condition"].map({
    "Base-Instruct + Prompt": review_lookup.get("Base-Instruct reviewer", 0),
    "Base-Instruct + 4-Shot": review_lookup.get("Base-Instruct reviewer", 0),
    "Base-Thinking + Prompt": review_lookup.get("Base-Thinking reviewer", 0),
    "LoRA-only": review_lookup.get("LoRA reviewer", 0),
    # Full-Project 的幕後 reviewer 就是同一顆 Base-Thinking GGUF。
    "Full-Project": review_lookup.get("Base-Thinking reviewer", 0),
}).fillna(0)
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
panels = [
    ("scenario_pass", "Behavioral-contract reliability", (0, 1.05)),
    ("turn3_retention", "Turn-3 style retention", (0, 1.05)),
    ("mean_input_tokens", "Per-request input-token burden", None),
    ("operational_success", "Operational reviewer success", (0, 1.05)),
]
for ax, (metric, title, ylim) in zip(axes.flat, panels):
    sns.barplot(data=dashboard, x="condition", y=metric, order=order, ax=ax)
    ax.set_title(title); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=23)
    if ylim: ax.set_ylim(*ylim)
fig.suptitle("Project value: specialization + state control + efficient prompting + verified review", y=1.01)
fig.tight_layout(); fig.savefig(RESULT_DIR / "05_project_value_dashboard.png", dpi=180, bbox_inches="tight")
plt.show()

print("輸出資料夾：", RESULT_DIR)


## 7. 產生匿名盲評表（必要）

自動規則只能判格式與明確違規，不能完整判斷提示是否真的有用、數學回饋是否精確、語氣是否尊重學生。請找至少 2 位不知道條件對應關係的評分者，為每筆單輪回答評：

- `style_fidelity_1to5`
- `math_accuracy_1to5`
- `usefulness_1to5`

Thinking 若沒有有效最終回答也要保留在盲評表，不能事後刪除。


In [ ]:
rng = random.Random(SEED)
blind_rows, key_rows = [], []
for (pid, scenario), group in single.groupby(["problem_id", "scenario"], sort=True):
    rows = group.to_dict("records")
    rng.shuffle(rows)
    for idx, row in enumerate(rows, 1):
        blind_id = f"{pid}-{scenario}-R{idx}"
        blind_rows.append({
            "blind_id": blind_id,
            "problem_id": pid,
            "scenario": scenario,
            "student_text": row["student_text"],
            "assistant_response": row["response"],
            "style_fidelity_1to5": "",
            "math_accuracy_1to5": "",
            "usefulness_1to5": "",
            "notes": "",
        })
        key_rows.append({"blind_id": blind_id, "condition": row["condition"]})

blind_df = pd.DataFrame(blind_rows)
key_df = pd.DataFrame(key_rows)
blind_df.to_csv(RESULT_DIR / "blind_scoring_sheet.csv", index=False, encoding="utf-8-sig")
key_df.to_csv(RESULT_DIR / "blind_key_DO_NOT_OPEN_BEFORE_SCORING.csv", index=False, encoding="utf-8-sig")
display(blind_df.head(8))
print("已建立匿名評分表與獨立解盲 key。")


## 8. 如何用這份結果回答教授

建議先說清楚專案定位：

> 兩顆基礎模型擁有「可能做出引導」的能力；本專案處理的是可靠度工程，將風格專門化、對話狀態、邊界守衛與數學複核拆成可量測元件。目標不是證明 Base 模型完全不會教，而是證明在壓力、多輪與錯誤草稿下，專案是否更穩定、成本是否更可預測。

四張證據的對應方式：

1. `01_contract_by_scenario.png`：回答「會引導」與「穩定遵守契約」的差別。
2. `02_turn_stress_and_phase.png`：回答 Prompt 漂移，以及狀態機是否真的在第三次卡住時接管。
3. `03_efficiency_and_token_tax.png`：回答 Few-shot 每次重送的 token 負擔；延遲必須連同輸出 token 一起解讀。
4. `04_review_operational_success.png`：回答 Thinking 為何適合幕後複核；不能只看語意命中，還要看 JSON 是否可解析。

`05_project_value_dashboard.png` 可作為簡報總覽，但若任何構面沒有改善，就應把它呈現為目前待修正的工程缺口，而不是隱藏。正式簡報再補上兩位盲評者的平均與一致性。


In [ ]:
# 結果已保存在 Google Drive；需要時再下載 zip，不必在執行中維持 Colab 連線。
import shutil
archive_base = PROJECT_CONTAINER / "professor_ablation_results_v2"
archive = shutil.make_archive(str(archive_base), "zip", RESULT_DIR)
print("已永久保存到 Google Drive：", archive)
try:
    from google.colab import files
    # 需要立即下載到本機時取消下一行註解：
    # files.download(archive)
except ImportError:
    pass
